In [ ]:
 !pip install datasets transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "

In [ ]:

from datasets import load_dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# 2. Load dataset
dataset = load_dataset("go_emotions")

# 3. Mapping ke Ekman 7
ekman_map = {
    'admiration': 'joy', 'amusement': 'joy', 'anger': 'anger', 'annoyance': 'anger',
    'approval': 'joy', 'caring': 'joy', 'confusion': 'surprise', 'curiosity': 'surprise',
    'desire': 'joy', 'disappointment': 'sadness', 'disapproval': 'disgust', 'disgust': 'disgust',
    'embarrassment': 'sadness', 'excitement': 'joy', 'fear': 'fear', 'gratitude': 'joy',
    'grief': 'sadness', 'joy': 'joy', 'love': 'joy', 'nervousness': 'fear', 'neutral': 'neutral',
    'optimism': 'joy', 'pride': 'joy', 'realization': 'surprise', 'relief': 'joy',
    'remorse': 'sadness', 'sadness': 'sadness', 'surprise': 'surprise'
}
ekman_labels = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/350k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
# Ambil referensi konverter dari fitur
label_feature = dataset['train'].features['labels'].feature

def map_labels(example):
    labels = [label_feature.int2str(l) for l in example['labels']]
    mapped = [ekman_map[l] for l in labels if l in ekman_map]
    example['ekman'] = mapped[0] if mapped else None
    return example

for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(map_labels)
    dataset[split] = dataset[split].filter(lambda x: x['ekman'] is not None)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
le = LabelEncoder()
le.fit(ekman_labels)

def encode_labels(example):
    example['label'] = le.transform([example['ekman']])[0]
    return example

for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(encode_labels)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(tokenize, batched=True)

# 6. Format
for split in ['train', 'validation', 'test']:
    dataset[split].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
# 7. Load model
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=7)

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    report_to="none"  # nonaktifkan wandb/tensorboard jika tidak dipakai
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-7-a7aebfee19db>:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.989600,0.873388,0.679690
2,0.808300,0.864245,0.686878
3,0.718400,0.877002,0.677110
4,0.632700,0.907825,0.682455
5,0.548800,0.978115,0.673240
6,0.472800,1.019112,0.671581


In [ ]:
# Evaluation on test set
preds_output = trainer.predict(dataset['test'])
y_true = preds_output.label_ids
y_pred = np.argmax(preds_output.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, xticklabels=le.classes_, yticklabels=le.classes_, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix on Test Set")
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
import torch
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# Fit LabelEncoder dari dataset
label_encoder = LabelEncoder()
label_encoder.fit(dataset['train']['ekman'])

# Load model & tokenizer
model_path = "/content/results/checkpoint-8142"  # atau "./results" jika hanya 1 checkpoint
model = RobertaForSequenceClassification.from_pretrained(model_path, num_labels=7)
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# Fungsi prediksi
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=-1).item()
    emotion_labels = label_encoder.inverse_transform([predicted_class])
    return emotion_labels[0]

# Contoh diary
diary_text = """
Today I feel very confused and don't know what to do.
But I'm also happy because finally my project is finished.
My friend didn't reply to my messages at all, which made me upset.
"""

# Inference per kalimat
sentences = sent_tokenize(diary_text)
for sentence in sentences:
    emotion = predict_emotion(sentence)
    print(f"Kalimat: '{sentence}' -> Prediksi Emosi: {emotion}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Kalimat: '
Today I feel very confused and don't know what to do.' -> Prediksi Emosi: surprise
Kalimat: 'But I'm also happy because finally my project is finished.' -> Prediksi Emosi: joy
Kalimat: 'My friend didn't reply to my messages at all, which made me upset.' -> Prediksi Emosi: sadness


In [ ]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
pip install --upgrade huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:

trainer.push_to_hub("Finetuned RoBERTa on Ekman GoEmotions")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.24k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/FadQ/results/commit/51df8d9df4ca31de41a269845720b7c42ea547eb', commit_message='Finetuned RoBERTa on Ekman GoEmotions', commit_description='', oid='51df8d9df4ca31de41a269845720b7c42ea547eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FadQ/results', endpoint='https://huggingface.co', repo_type='model', repo_id='FadQ/results'), pr_revision=None, pr_num=None)




Kolom A | Kolom B | Label

  a         b         topik